[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced/blob/main/CH05/05_04_end/notebooks/05_04_fine_tune_challenge.ipynb)

# 05_04 Challenge: Fine-tune the support assistant

Choose your own LoRA and training settings, train the Ridgeline assistant, and prove it learned. The self-check near the end tells you if you're done.

Before you start, choose **Runtime > Change runtime type > T4 GPU**.

In [ ]:
%%capture
# Install Unsloth. This takes a minute or two.
!pip install unsloth

In [ ]:
# Pull the course files into this Colab session and load the Ridgeline helpers.
!git clone --quiet https://github.com/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced.git course 2>/dev/null || echo "Course files already downloaded."

import sys
sys.path.insert(0, "course/CH05")

import ridgeline

## Load the model

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length=2048,
    load_in_4bit=True,
)

## Load the data and summarize it

In [ ]:
from datasets import load_dataset

# Use the course checkpoint, or upload your own files from Chapter 4 and change these paths.
TRAIN_FILE = "course/CH05/data/train.jsonl"
EVAL_FILE = "course/CH05/data/eval.jsonl"

dataset = load_dataset("json", data_files={"train": TRAIN_FILE, "eval": EVAL_FILE})


def to_text(example):
    """Render each conversation with the model's own chat template."""
    return {"text": ridgeline.chat_text(tokenizer, example["messages"])}


dataset = dataset.map(to_text, remove_columns=dataset["train"].column_names)
print(dataset)

In [ ]:
import json
from collections import Counter

rows = [json.loads(line) for line in open(TRAIN_FILE)]
lengths = [len(tokenizer(text)["input_ids"]) for text in dataset["train"]["text"]]
counts = Counter(row["category"] for row in rows)

summary = "\n".join([
    "# Dataset summary",
    "",
    f"- Training examples: {len(dataset['train'])}",
    f"- Evaluation examples: {len(dataset['eval'])}",
    f"- Tokens per training example: shortest {min(lengths)}, average {sum(lengths) // len(lengths)}, longest {max(lengths)}",
    "- Training examples per category:",
    *[f"  - {category}: {count}" for category, count in sorted(counts.items())],
])
open("dataset-summary.md", "w").write(summary)
print(summary)

## Optional: get a second opinion on your settings

Before you train, ask an AI assistant to critique your plan. Copy the dataset summary above and the contents of `course/CH05/challenge/training-goals.md` into a chat with an assistant like Claude, along with this request:

> Here's my dataset and my training goals for a LoRA fine-tune. Recommend values for rank, lora_alpha, learning rate, epochs, and batch size, and explain the tradeoff behind each one. Then tell me what you'd watch for in the loss curves.

Decide which advice to take. After training, check whether the results matched its predictions.

## Add your LoRA adapter

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                   # rank: how much room the adapter has to learn
    lora_alpha=16,          # scaling: how strongly the adapter's update counts
    lora_dropout=0,         # Unsloth is optimized for 0
    target_modules=[        # the layers that get adapters
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention
        "gate_proj", "up_proj", "down_proj",      # feed-forward
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",       # saves memory
    random_state=3407,
)
model.print_trainable_parameters()

## Choose your training settings

In [ ]:
from trl import SFTConfig
from unsloth import is_bfloat16_supported

config = SFTConfig(
    output_dir="outputs",
    dataset_text_field="text",
    max_length=2048,
    per_device_train_batch_size=2,     # examples per step on the GPU
    gradient_accumulation_steps=4,     # 2 x 4 = 8 examples per update
    num_train_epochs=2,               # passes through the training data
    learning_rate=2e-4,              # how big each update step is
    warmup_steps=5,                    # ease into the learning rate
    lr_scheduler_type="linear",
    optim="adamw_8bit",                # memory-efficient optimizer
    weight_decay=0.01,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=20,
    per_device_eval_batch_size=4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    seed=3407,
    report_to="none",
)

## Build the trainer and train

In [ ]:
from trl import SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["eval"],
    args=config,
)

# Learn only from the assistant's reply, not from the system prompt or the ticket.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

trainer.evaluate()
stats = trainer.train()
print(f"Training took {stats.metrics['train_runtime'] / 60:.1f} minutes")

## Read the loss curves

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_points = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_points = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.plot(*zip(*train_points), label="training loss")
plt.plot(*zip(*eval_points), marker="o", label="evaluation loss")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()

## Test on unseen tickets

In [ ]:
FastLanguageModel.for_inference(model)
tickets = {t["id"]: t for t in ridgeline.load_tickets()}

results = []
for ticket_id in ["RT-001", "RT-008", "RT-017"]:
    ticket = tickets[ticket_id]
    # No voice rules in the prompt. The model has to have learned them.
    messages = ridgeline.build_messages(ticket, include_voice_rules=False)
    reply = ridgeline.generate(model, tokenizer, messages)
    print(f"--- {ticket_id}: {ticket['subject']} ---")
    print(reply, "\n")
    result = ridgeline.check_reply(ticket, reply)
    ridgeline.print_check(result)
    results.append(result)
    print()

## Check your work

In [ ]:
# Check your work. Each line prints PASS or FAIL.

def check(label, passed):
    print(f"{'PASS' if passed else 'FAIL'}  {label}")

evals = [h["eval_loss"] for h in trainer.state.log_history if "eval_loss" in h]
header_ok = ["missing or broken Category/Escalate header" not in r["failures"] for r in results]

check("Training ran to the last step", trainer.state.global_step == trainer.state.max_steps)
check("Evaluation loss was measured before and during training", len(evals) >= 2)
check("Final evaluation loss is lower than the starting loss", len(evals) >= 2 and evals[-1] < evals[0])
check("Evaluation loss stayed within 5 percent of its lowest point", bool(evals) and evals[-1] <= min(evals) * 1.05)
check("At least 2 of 3 test replies use the Category and Escalate format", sum(header_ok) >= 2)
check("At least 2 of 3 test tickets got the right triage",
      sum(r["category_ok"] and r["escalate_ok"] for r in results) >= 2)

## Save your adapter to Google Drive

Chapter 6 evaluates and deploys this adapter.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

ADAPTER_DIR = "/content/drive/MyDrive/ridgeline-lora"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Saved your LoRA adapter to {ADAPTER_DIR}")